# Library Imports

In [ ]:
!pip install neuralforecast hyperopt

# Data prep & all of them
!pip install utilsforecast
import pandas as pd
import matplotlib.pyplot as plt
from utilsforecast.plotting import plot_series
from utilsforecast.losses import bias, rmse, mae, mape, bias
from utilsforecast.evaluation import evaluate


from neuralforecast import NeuralForecast
from ray import tune
from neuralforecast.auto import AutoNBEATS


# Data Preparation (this is shortened from first version)

In [ ]:
df_base = pd.read_parquet('/content/sample_hotels-1.parquet')
df_base.head()

,unique_id,ds,holiday_flag,target_day,target_month,target_year,location_type,hotel_type,y,otb_1,...,otb_51,otb_52,otb_53,otb_54,otb_55,otb_56,otb_57,otb_58,otb_59,otb_60
1430,hotel_0,2022-01-01,no,Sat,Jan,2022,NonSuburban,Resorts & Destinations,0.975309,0.679012,...,0.197531,0.197531,0.197531,0.185185,0.160494,0.160494,0.160494,0.160494,0.160494,0.160494
1431,hotel_0,2022-01-02,no,Sun,Jan,2022,NonSuburban,Resorts & Destinations,0.493827,0.308642,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.061728,0.061728,0.049383
1432,hotel_0,2022-01-03,no,Mon,Jan,2022,NonSuburban,Resorts & Destinations,0.456790,0.358025,...,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691,0.024691
1433,hotel_0,2022-01-04,no,Tue,Jan,2022,NonSuburban,Resorts & Destinations,0.592593,0.419753,...,0.074074,0.074074,0.074074,0.074074,0.061728,0.061728,0.037037,0.037037,0.024691,0.024691
1434,hotel_0,2022-01-05,no,Wed,Jan,2022,NonSuburban,Resorts & Destinations,0.530864,0.407407,...,0.074074,0.074074,0.074074,0.074074,0.074074,0.049383,0.049383,0.024691,0.024691,0.012346


Drop unimformative variable

In [ ]:
df_base = df_base.drop(columns=['target_year'])

Dummies

In [ ]:
 # Convert holiday flag to boolean manually
 df_base['holiday_flag'] = df_base['holiday_flag'].astype('bool')

In [ ]:
cat_cols = [
    'target_day',
    'target_month',
    'location_type',
    'hotel_type'
]

for col in cat_cols:
    df_base[col] = df_base[col].astype('category')

In [ ]:
df_base = pd.get_dummies(df_base, columns=cat_cols, drop_first=True)

OTB

In [ ]:
# Drop all OTB values before otb_28 because that information wouldn't actually be available at the time of forecasting
# By only keeping otb_28 and beyond, I ensure the model doesn't "cheat" by looking at data from inside the 28-day period
columns_to_drop = [f'otb_{i}' for i in range(1, 28)]
df_base = df_base.drop(columns=columns_to_drop)
display(df_base.head())

,unique_id,ds,y,otb_28,otb_29,otb_30,otb_31,otb_32,otb_33,otb_34,...,target_month_Jun,target_month_Mar,target_month_May,target_month_Nov,target_month_Oct,target_month_Sep,location_type_NonSuburban,hotel_type_Key Central Business District,hotel_type_Other High Leisure Mix,hotel_type_Resorts & Destinations
1430,hotel_0,2022-01-01,0.975309,0.296296,0.283951,0.296296,0.296296,0.283951,0.259259,0.234568,...,False,False,False,False,False,False,True,False,False,True
1431,hotel_0,2022-01-02,0.493827,0.086420,0.086420,0.086420,0.086420,0.086420,0.086420,0.086420,...,False,False,False,False,False,False,True,False,False,True
1432,hotel_0,2022-01-03,0.456790,0.061728,0.061728,0.061728,0.061728,0.061728,0.061728,0.049383,...,False,False,False,False,False,False,True,False,False,True
1433,hotel_0,2022-01-04,0.592593,0.111111,0.111111,0.111111,0.111111,0.098765,0.098765,0.098765,...,False,False,False,False,False,False,True,False,False,True
1434,hotel_0,2022-01-05,0.530864,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,...,False,False,False,False,False,False,True,False,False,True


Drop Hotels

In [ ]:
hotels_to_drop = ['hotel_28', 'hotel_77']
df_base = df_base[df_base['unique_id'].isin(hotels_to_drop) == False]

Test/Train and Pred/No Pred Split

In [ ]:
# cutoff
cutoff = '2023-05-31'

#train/test split
train = df_base[df_base['ds'] <= cutoff]
test = df_base[df_base['ds'] > cutoff]

# full df no pred
df_no_pred = df_base[['unique_id', 'ds', 'y']]

# train
train_base = train[['unique_id', 'ds', 'y']] # Nixtila format dataset
train_ml = train.copy() # full dataset for ML

test_base = test[['unique_id', 'ds', 'y']]
test_ml = test.copy()

# AutoNBEATS Cross Validation and Evaluation

In [ ]:
# Extract the default hyperparameter settings for AutoNBEATS
nbeats_config = AutoNBEATS.get_default_config(h=28, backend="ray")

# Customize max_steps (100 for now, 1000 for final)
nbeats_config["max_steps"] = tune.choice([100])

# Using a range for the random seed during the search phase
nbeats_config["random_seed"] = tune.randint(1, 10)

In [ ]:
# Define model
auto_nb_model = [
    AutoNBEATS(
        h=28,
        config=nbeats_config,
        num_samples = 20
    )
]

In [ ]:
# Create model object
n_beat = NeuralForecast(
    models=auto_nb_model,
    freq='D')

In [ ]:
# Cross Validation
cross_autonb = n_beat.cross_validation(
    df = df_base,
    step_size = 28,
    n_windows = 5
)

2026-05-05 12:39:30,402	INFO worker.py:2012 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(
2026-05-05 12:39:35,618	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `Tuner(...)`.


+--------------------------------------------------------------------+
| Configuration for experiment     _train_tune_2026-05-05_12-39-18   |
+--------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator             |
| Scheduler                        FIFOScheduler                     |
| Number of trials                 20                                |
+--------------------------------------------------------------------+

View detailed results here: /root/ray_results/_train_tune_2026-05-05_12-39-18
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2026-05-05_12-39-18_962880_3837/artifacts/2026-05-05_12-39-35/_train_tune_2026-05-05_12-39-18/driver_artifacts`


(_train_tune pid=4546) /usr/local/lib/python3.12/dist-packages/ray/tune/integration/pytorch_lightning.py:198: `ray.tune.integration.pytorch_lightning.TuneReportCallback` is deprecated. Use `ray.tune.integration.pytorch_lightning.TuneReportCheckpointCallback` instead.
(_train_tune pid=4546) Seed set to 2
(_train_tune pid=4546) GPU available: False, used: False
(_train_tune pid=4546) TPU available: False, using: 0 TPU cores
(_train_tune pid=4546) 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
(_train_tune pid=4546) 2026-05-05 12:39:53.510685: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
(_train_tune pid=4546) 2026-05-05 12:39:53.591433: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use ava

(_train_tune pid=4546) ┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
(_train_tune pid=4546) ┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
(_train_tune pid=4546) ┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
(_train_tune pid=4546) │ 0 │ loss         │ MAE           │      0 │ train │     0 │
(_train_tune pid=4546) │ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
(_train_tune pid=4546) │ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
(_train_tune pid=4546) │ 3 │ blocks       │ ModuleList    │  2.7 M │ train │     0 │
(_train_tune pid=4546) └───┴──────────────┴───────────────┴────────┴───────┴───────┘
(_train_tune pid=4546) Trainable params: 2.7 M                                                         
(_train_tune pid=4546) Non-trainable params: 9.6 K                                                     
(_train_tune pid=4546) Total params: 2.7 M                                                             
(_train_

(_train_tune pid=4546) /usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
(_train_tune pid=4546) `Trainer.fit` stopped: `max_steps=100` reached.


...

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

In [ ]:
# save as parquet just incase
cross_autonb.to_parquet('cross_autonb.parquet', index=False)

In [ ]:
# Evaluate the model
eval_autonbeats = evaluate(
    df=cross_autonb,
    metrics=[bias, mae, rmse, mape],
    models=['AutoNBEATS'])

In [ ]:
# Drop identifiers and average metrics across all series and cutoffs for easier metrics comparison
eval_autonbeats = eval_autonbeats.drop(columns=['cutoff']).groupby(['unique_id', 'metric']).mean().reset_index(inplace=False)
eval_autonbeats

# Save evaluation as parquete
eval_autonbeats.to_parquet('eval_autonbeats_1.parquet', index=False)

In [ ]:
eval_autonbeats.head()

,unique_id,metric,AutoNBEATS
0,hotel_0,bias,-0.004609
1,hotel_0,mae,0.161878
2,hotel_0,mape,0.261339
3,hotel_0,rmse,0.195913
4,hotel_105,bias,-0.090817
